In [0]:
# Databricks notebook source
"""Run data quality checks on the bronze PSA population table."""

from __future__ import annotations

import sys

from pyspark.sql import Column, SparkSession
from pyspark.sql.functions import col, count, expr, lit, trim

# COMMAND ----------

BRONZE_TABLE = "ahon.bronze.psa_population_raw"
MERGE_KEYS = ["geographic_location", "census_year"]
NULL_CHECK_COLS = (
    "total_population",
    "urban_population",
    "percent_urban",
    "census_year",
)
NUMERIC_COLS = ("total_population", "urban_population", "percent_urban")
INFO_CHECKS = {"total_rows"}  # reported only, never counted as a failure

# COMMAND ----------


def report(message: str) -> None:
    """Write a line to stdout, which Databricks shows as cell output."""
    sys.stdout.write(f"{message}\n")


def non_numeric(column: str) -> Column:
    """Return a condition that is true when a non-null value is not a number."""
    return col(column).isNotNull() & expr(f"try_cast({column} AS DOUBLE)").isNull()


def build_row_checks() -> dict[str, Column]:
    """Return the row-level checks, each true for a failing row.

    The conditions are built inside a function, not at import time, because
    pyspark's col() needs an active Spark session.
    """
    checks: dict[str, Column] = {
        "null_geographic_location": col("geographic_location").isNull(),
        "empty_geographic_location": (
            col("geographic_location").isNotNull()
            & (trim(col("geographic_location")) == "")
        ),
    }
    for name in NULL_CHECK_COLS:
        checks[f"null_{name}"] = col(name).isNull()
    for name in NUMERIC_COLS:
        checks[f"non_numeric_{name}"] = non_numeric(name)
    return checks


def run_bronze_dq_checks(spark: SparkSession) -> dict[str, int]:
    """Run data quality checks on the bronze table and print a summary.

    Bronze stores population columns as STRING, so the checks include
    non-numeric value detection in addition to nulls and duplicates. Returns a
    dict mapping each check name to its failing row count, where 0 means the
    check passed. The total row count is included for reference.
    """
    bronze = spark.table(BRONZE_TABLE)

    results = {"total_rows": bronze.count()}
    for name, failing in build_row_checks().items():
        results[name] = bronze.filter(failing).count()
    results["duplicate_merge_keys"] = (
        bronze.groupBy(*MERGE_KEYS)
        .agg(count(lit(1)).alias("n"))
        .filter(col("n") > 1)
        .count()
    )

    report("-- Bronze DQ Checks --")
    for check, value in results.items():
        status = "PASS" if check in INFO_CHECKS or value == 0 else "FAIL"
        report(f"  {check:40s} {value:>10,}  {status}")
    return results


def main() -> None:
    """Run bronze DQ checks and print the summary."""
    spark = SparkSession.builder.getOrCreate()
    run_bronze_dq_checks(spark)


# COMMAND ----------

if __name__ == "__main__":
    main()